# 03. T1 분기점 검증 — ③ 키워드·소비 심리 신호 / ④ E12 vs E13 / ⑥ 국면 변화

**배경** (`train.ipynb` 실험 기록)
| 실험 | 구성 | CV 평균±std | 평균−std | 비고 |
| --- | --- | --- | --- | --- |
| E12_t1_blend | E8 RF + LR(미국 지수·환율 diff) | 0.621±0.019 | 0.603 | |
| E13_t1_blend | E12 + LR에 유가 diff·**금융 키워드 심리** | 0.637±0.033 | 0.604 | 후보 ~15개 중 선택 |

**이 노트북이 답하는 질문**
| # | 질문 | 위치 |
| --- | --- | --- |
| ③ | 키워드 심리(`kefin_swd`)·소비 분위기(`samewd_grp`) 신호는 **우연이 아닌가? 시기와 무관하게 유지되나? 미국장과 다른 정보인가?** | §1 |
| ④ | E13의 +0.016은 **잡음·선택 편향을 넘는 개선인가?** | §2 |
| ⑥ | 국면이 바뀌는 T1에서 **최근 데이터만 쓰거나 최근에 가중**을 주면 나아지나? | §4 |
| + | Prophet 단서: T1에 **1년 주기**가 있나? | §5 |
| + | Public에서 E8이 높았던 이유 + 변동성에 따라 모델 전환 | §6 |

**원칙**
- Train만 사용한다. 모든 피처는 `train.ipynb`와 **같은 코드**(과거 값만)로 만든다.
- 시계열은 하루하루가 독립이 아니다(자기상관). 그래서 보통의 검정(독립 가정) 대신 **순환 이동 검정**과 **블록 부트스트랩**을 쓴다.
  - 순환 이동: 피처를 k칸 통째로 돌려 붙인다 → 피처 자체의 흐름(자기상관)은 그대로, Target과의 시점 관계만 끊긴다 = "이 정도 흐름을 가진 아무 변수"의 기준선.
  - 블록 부트스트랩: 20일씩 연속 구간을 뽑아 다시 이어 붙인다 → 붙어 있는 날들의 의존성을 보존한 채 표본 흔들림을 잰다.
- **판정 기준은 결과를 보기 전에 정해 둔다** (각 절 첫머리). 결과를 보고 기준을 바꾸면 그 자체가 과적합이다.

In [ ]:
# train.ipynb §1~§4(데이터·피처 블록·모델·E0~E13 실험 등록)를 조용히 불러온다.
# 실험은 experiments.csv 캐시와 설정이 같으면 학습 없이 건너뛴다 → 몇 초, 기록 파일은 바뀌지 않아야 정상.
import json, os, contextlib, io
import matplotlib.pyplot as plt
from IPython.display import display

_mt = os.path.getmtime('experiments.csv')
_nb = json.load(open('train.ipynb', encoding='utf-8'))
_src = [''.join(c['source']) for c in _nb['cells']]
_stop = next(i for i, (c, s) in enumerate(zip(_nb['cells'], _src)) if c['cell_type'] == 'markdown' and s.startswith('## 5. 최종'))
_show, plt.show = plt.show, (lambda *a, **k: plt.close('all'))
globals()['display'] = lambda *a, **k: None
with contextlib.redirect_stdout(io.StringIO()):
    for c, s in zip(_nb['cells'][:_stop], _src[:_stop]):
        if c['cell_type'] == 'code':
            exec(s, globals())
plt.show = _show
from IPython.display import display
plt.close('all')
assert os.path.getmtime('experiments.csv') == _mt, 'experiments.csv가 바뀜 = train.ipynb 설정이 캐시와 다름'
print('불러옴 | T1 실험:', sorted(n for n, t in REG if t == 'Target_1'))

In [ ]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(SEED)
y1 = Y['Target_1']                        # Train 924일
yv = y1.values
n = len(y1)
FOLDS = list(TSCV.split(np.arange(n)))    # train.ipynb와 같은 5-fold, gap=3
VA = np.concatenate([va for _, va in FOLDS])   # 검증 구간 전체(OOF가 있는 날)
VA_START = [va[0] for _, va in FOLDS]
tr_ = lambda D: D[is_tr].reset_index(drop=True)

def auc(x, y):
    x, y = np.asarray(x, float), np.asarray(y)
    m = ~np.isnan(x)
    return roc_auc_score(y[m], x[m]) if m.sum() > 20 and len(set(y[m])) == 2 else np.nan

def circ_pvalue(x, y, margin=30):
    # 순환 이동 검정 (양측): |AUC-0.5|가 'k칸 돌린 같은 피처'들보다 큰 비율
    x = np.asarray(x, float)
    obs = abs(auc(x, y) - .5)
    null = np.array([abs(auc(np.roll(x, k), y) - .5) for k in range(margin, len(x) - margin)])
    return (1 + (null >= obs).sum()) / (1 + len(null))

def block_idx(m, L=20):
    # 이동 블록 부트스트랩 인덱스: 길이 L 연속 구간을 무작위로 뽑아 이어 붙임
    starts = rng.integers(0, m - L + 1, size=int(np.ceil(m / L)))
    return np.concatenate([np.arange(s, s + L) for s in starts])[:m]

def oof(model, Xd):
    # fold마다 과거로 학습 → 그 fold 검증 구간 예측. 검증 구간 밖은 NaN
    p = np.full(n, np.nan)
    for tr, va in FOLDS:
        p[va] = clone(model).fit(Xd.iloc[tr], y1.iloc[tr]).predict_proba(Xd.iloc[va])[:, 1]
    return p

fold_mean = lambda p: np.mean([auc(p[va], yv[va]) for _, va in FOLDS])
print('검증 구간', len(VA), '일 | fold 검증 시작 위치', VA_START)

## 1. ③ 키워드 심리·소비 분위기 신호는 믿을 만한가

**후보**: `kefin_swd`(금융 키워드 6개의 '같은 요일 4주 평균 대비' 중앙값, E13에서 추가) · 그 구성 키워드 6개 · `samewd_grp`(쇼핑L1·L2·키워드 군별 sameWD 중앙값, E2부터 RF에 사용)
**참고 기준선**: 미국 지수 diff 평균(원인이 분명한 신호), 환율·유가 diff 평균

**판정 기준 (미리 정함)** — 신호별로 아래를 모두 만족하면 **"믿을 만함"**, 하나라도 못 넘으면 **"우연 가능성 있음"**
1. 순환 이동 검정 p < 0.05
2. 블록 부트스트랩 95% 구간이 0.5를 포함하지 않음
3. 앞 절반·뒤 절반에서 방향이 같음
4. 150일 이동 창 AUC가 전체 방향과 같은 쪽에 있는 비율 ≥ 70%
5. (`kefin_swd`만) LR에서 5개 fold 모두 계수 부호가 같고, 미국장 신호를 고정한 뒤에도 효과가 남음 (§1-3)

> ⚠️ `kefin_swd`의 키워드 6개는 Train 전체를 보고 고른 것이라, §1-1의 단일 AUC는 다소 낙관적이다. 그래서 2·4번(시기별 유지)과 §2(선택 편향 검정)를 같이 본다.

### 1-1. 단일 신호 검정표 — AUC, 앞/뒤 절반, 블록 부트스트랩 구간, 순환 이동 p

In [ ]:
ke_fin_swd = tr_(ke_fin / same_wd_mean(ke_fin)).add_suffix('_swd')
CAND = pd.concat([
    tr_(BLOCKS['kefin_swd']), ke_fin_swd, tr_(BLOCKS['samewd_grp']),
    tr_(BLOCKS['us_diff']).mean(axis=1).rename('[참고] 미국지수 diff'),
    tr_(BLOCKS['fx_diff']).mean(axis=1).rename('[참고] 환율 diff'),
    tr_(BLOCKS['oil_diff']).mean(axis=1).rename('[참고] 유가 diff'),
], axis=1).replace([np.inf, -np.inf], np.nan)

half = n // 2
rows = []
for c in CAND:
    x = CAND[c].values
    boot = [auc(x[i], yv[i]) for i in (block_idx(n) for _ in range(500))]
    rows.append({'변수': c, 'AUC': auc(x, yv), '앞 절반': auc(x[:half], yv[:half]), '뒤 절반': auc(x[half:], yv[half:]),
                 'CI 2.5%': np.nanpercentile(boot, 2.5), 'CI 97.5%': np.nanpercentile(boot, 97.5),
                 '순환이동 p': circ_pvalue(x, yv)})
sig = pd.DataFrame(rows).set_index('변수')
sig['방향 일치'] = np.sign(sig['앞 절반'] - .5) == np.sign(sig['뒤 절반'] - .5)
sig['CI가 0.5 제외'] = (sig['CI 2.5%'] > .5) | (sig['CI 97.5%'] < .5)
display(sig.round(3).style.format(precision=3).background_gradient(subset=['AUC'], cmap='RdBu_r', vmin=.35, vmax=.65))

#### 👀 표 읽는 법
- **AUC**: 0.5 = 무관, 0.5보다 크면 '값이 클수록 3일 뒤 상승', 작으면 반대.
- **순환이동 p**: 같은 흐름을 가진 '아무 변수'가 이만한 AUC를 낼 확률. 키워드처럼 천천히 움직이는 변수는 우연히도 AUC가 크게 나오기 쉬워서, 이 p가 보통 검정보다 훨씬 보수적이다.
- **CI**: 시기를 바꿔 뽑았을 때 AUC가 흔들리는 범위. 0.5를 걸치면 "시기에 따라 무관할 수도 있다".

### 1-2. 시기별 유지 — 150일 이동 창 AUC (점선 = 무관 0.5, 회색 띠 = 순수 잡음 95% 범위)

In [ ]:
W, STEP = 150, 10
starts = np.arange(0, n - W + 1, STEP)
roll = {c: np.array([auc(CAND[c].values[s:s + W], yv[s:s + W]) for s in starts]) for c in CAND}
band = 1.96 * np.sqrt((W + 1) / (12 * (W / 2) ** 2))      # 상승/하락 반반일 때 AUC 표준오차 근사

ROLL_SHARE = {c: np.mean(np.sign(roll[c] - .5) == np.sign(sig.loc[c, 'AUC'] - .5)) for c in CAND}
groups = {'금융 키워드 심리 (E13 LR에 추가)': ['kefin_swd_med', 'KE_부동산_swd', 'KE_아파트_swd', 'KE_대출_swd'],
          '소비 분위기 samewd_grp (RF에 사용)': ['SH1_sameWD_med', 'SH2_sameWD_med', 'KE_sameWD_med']}
mid = starts + W // 2
fig, ax = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for a, (title, cs) in zip(ax, groups.items()):
    a.axhspan(.5 - band, .5 + band, color='0.9', label='잡음 95% 범위')
    a.plot(mid, roll['[참고] 미국지수 diff'], c='k', lw=2.5, label='[참고] 미국지수 diff')
    for c in cs:
        a.plot(mid, roll[c], lw=2 if c in ('kefin_swd_med', 'SH2_sameWD_med') else 1, label=f'{c} (같은 방향 {ROLL_SHARE[c]:.0%})')
    a.axhline(.5, c='k', ls=':', lw=.8)
    for s in VA_START: a.axvline(s, c='tomato', ls='--', lw=.7)
    a.set_title(f'{title} — 창 {W}일 AUC (빨간 점선 = CV fold 검증 시작)'); a.legend(fontsize=8, ncol=3, loc='lower left'); a.set_ylabel('AUC')
ax[1].set_xlabel('Time_step (창 중앙)')
plt.tight_layout(); plt.show()
pd.Series(ROLL_SHARE, name='전체 방향과 같은 창 비율').round(2).to_frame().T

#### 👀 그림 읽는 법
- 선이 **회색 띠 밖에 오래 머물수록** 진짜 신호. 띠 안에서 0.5를 오가면 그 시기엔 잡음과 구별되지 않는다.
- 검정 선(미국장)은 원인이 분명한 신호의 기준이다. 키워드·소비 선이 검정 선처럼 **전 구간 같은 쪽**에 있으면 믿을 만하고, **뒤쪽에서만** 튀면 최근 국면에 우연히 맞은 것일 수 있다.
- Private(Test 925~1124)은 그림 오른쪽 끝 다음 구간이다. **오른쪽 끝에서 신호가 살아 있는지**가 가장 중요하다.

### 1-3. 미국장과 다른 정보인가 — 상관 · 미국장 고정 후 효과 · fold별 LR 계수

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 5.5), gridspec_kw={'width_ratios': [1.1, 1, 1.3]})
# ① 순위 상관: 미국장·환율과 거의 무관해야 '새 정보'
cc = ['kefin_swd_med', 'SH1_sameWD_med', 'SH2_sameWD_med', 'KE_sameWD_med', '[참고] 미국지수 diff', '[참고] 환율 diff', '[참고] 유가 diff']
r = CAND[cc].corr('spearman')
im = ax[0].imshow(r, cmap='RdBu_r', vmin=-1, vmax=1)
ax[0].set_xticks(range(len(cc)), [c.replace('[참고] ', '') for c in cc], rotation=45, ha='right', fontsize=8)
ax[0].set_yticks(range(len(cc)), [c.replace('[참고] ', '') for c in cc], fontsize=8)
for i in range(len(cc)):
    for j in range(len(cc)): ax[0].text(j, i, f'{r.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax[0].set_title('① 순위 상관 (|r| 작을수록 서로 다른 정보)')

# ② 미국장 3분위를 고정한 채 키워드 심리 3분위별 T1 상승률
us_q = pd.qcut(CAND['[참고] 미국지수 diff'], 3, labels=['미국 약세', '중립', '미국 강세'])
ke_q = pd.qcut(CAND['kefin_swd_med'], 3, labels=['심리 낮음', '중간', '심리 높음'])
tab = y1.groupby([us_q, ke_q], observed=True).mean().unstack()
cnt = y1.groupby([us_q, ke_q], observed=True).size().unstack()
im = ax[1].imshow(tab, cmap='RdYlGn', vmin=.25, vmax=.75)
ax[1].set_xticks(range(3), tab.columns); ax[1].set_yticks(range(3), tab.index)
for i in range(3):
    for j in range(3): ax[1].text(j, i, f'{tab.iloc[i, j]:.2f}\n(n={cnt.iloc[i, j]})', ha='center', va='center', fontsize=9)
ax[1].set_title('② T1 상승률: 같은 미국장 안에서 키워드 심리별\n(행마다 왼→오 증가 = 미국장과 별개 효과)')

# ③ fold별 LR 계수 (E13 LR 부분과 같은 설정, 표준화 계수)
lr_X = tr_(make_X(list(LR13)))
lr_m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=0.01, max_iter=3000))
coef = pd.DataFrame({f'fold{k + 1}': clone(lr_m).fit(lr_X.iloc[tr], y1.iloc[tr])[-1].coef_[0] for k, (tr, _) in enumerate(FOLDS)},
                    index=[c.replace('_dlast', '').replace('FI_', '') for c in lr_X.columns])
coef.plot.barh(ax=ax[2], width=.8, colormap='viridis'); ax[2].axvline(0, c='k', lw=.8)
ax[2].set_title('③ fold별 LR 표준화 계수 (막대 5개가 같은 쪽 = 안정)'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

same_sign = (np.sign(coef).nunique(axis=1) == 1)
mono = (tab.diff(axis=1).iloc[:, 1:] > 0).all(axis=1)
print('계수 부호가 5개 fold에서 같은 변수:', list(coef.index[same_sign]))
print('미국장 3분위 각각에서 키워드 심리↑ → 상승률↑ (단조 증가):', dict(mono))

#### 👀 그림 읽는 법
- **①** 키워드 심리와 미국장·환율의 상관이 0에 가까우면 LR에 넣었을 때 **새 정보**를 더한다. 쇼핑 sameWD끼리는 높게 묶여 있어도 상관없다(이미 군별 1개로 압축함).
- **②** 각 행(같은 미국장 조건)에서 오른쪽으로 갈수록 상승률이 오르면, 키워드 효과가 미국장의 다른 표현이 아니라는 뜻이다. 칸마다 n≈100이라 ±0.1 정도 차이는 잡음일 수 있다.
- **③** fold1은 가장 짧은 과거(약 150일)로 학습하므로 계수가 작거나 흔들려도 이상하지 않다. **fold 2~5에서 부호가 유지되는지**가 핵심이다. 유가 3종은 서로 상관이 커서 부호가 엇갈릴 수 있다(공선성).

### 1-4. 금융 키워드 6개는 특별한가 — 키워드 전체의 '평소 대비 검색량' 공통 요인과 비교

In [ ]:
# kefin_swd는 6개 키워드를 Train 전체를 보고 고른 것. 아무 키워드나 같은 방식(sameWD)으로 봐도 비슷한 AUC면
# '금융 심리'가 아니라 '키워드 검색량 전반의 공통 요인'(RF 쪽 samewd_grp의 KE_sameWD_med와 같은 정보)이다.
ke_all = X0[cols('KE')]
ke_swd = tr_(ke_all / same_wd_mean(ke_all)).replace([np.inf, -np.inf], np.nan)
ke_auc = ke_swd.apply(lambda s: auc(s.values, yv)).dropna().sort_values()
fin = [f'{k}' for k in KE_FIN if k in ke_auc.index]
rest_med = tr_((ke_all.drop(columns=KE_FIN) / same_wd_mean(ke_all.drop(columns=KE_FIN))).replace([np.inf, -np.inf], np.nan).median(axis=1))

fig, ax = plt.subplots(1, 2, figsize=(18, 4.5))
ax[0].bar(range(len(ke_auc)), ke_auc - .5, color=['tomato' if k in fin else '0.7' for k in ke_auc.index])
ax[0].axhline(0, c='k', lw=.8); ax[0].set_xticks([]); ax[0].set_ylabel('AUC - 0.5')
ax[0].set_title(f'키워드 {len(ke_auc)}개 sameWD 단일 AUC (빨강 = 금융 6개) | 0.5 초과 {np.mean(ke_auc > .5):.0%}')
ax[1].scatter(CAND['kefin_swd_med'], rest_med, s=6, alpha=.4)
ax[1].set_xlabel('kefin_swd (금융 6개)'); ax[1].set_ylabel('나머지 키워드 sameWD 중앙값')
ax[1].set_title(f"금융 6개 vs 나머지 키워드: 순위 상관 {CAND['kefin_swd_med'].corr(rest_med, method='spearman'):.2f}")
plt.tight_layout(); plt.show()
display(pd.DataFrame({'AUC': [auc(CAND['kefin_swd_med'].values, yv), auc(rest_med.values, yv), auc(CAND['KE_sameWD_med'].values, yv)],
                      '순환이동 p': [circ_pvalue(CAND['kefin_swd_med'].values, yv), circ_pvalue(rest_med.values, yv), circ_pvalue(CAND['KE_sameWD_med'].values, yv)]},
                     index=['금융 6개 (kefin_swd)', '금융 제외 나머지 키워드', 'KE 전체 (samewd_grp, 이미 RF에 있음)']).round(3))

#### 👀 읽는 법
- 회색(금융 아닌 키워드)도 대부분 0.5 위에 있고 '나머지 키워드' AUC가 금융 6개와 비슷하면 → 신호의 정체는 **"사람들이 평소보다 검색을 많이 하는 날"이라는 공통 요인**이다. 금융 심리라는 해석은 약해지지만, 신호 자체가 우연이라는 뜻은 아니다(키워드 수십 개에서 같은 방향).
- 그 경우 E13의 개선은 '이미 RF에 있는 KE_sameWD_med를 **LR에도** 넣은 효과'에 가깝다. → 해석을 "검색 활동 공통 요인"으로 바꿔 보고서에 쓰는 것이 정직하다.

## 2. ④ E12에서 멈출까, E13까지 갈까

E13 − E12: CV 평균 +0.0155, fold 4/5 우위, 평균−std는 거의 같음(0.603 vs 0.604). 이 차이가 **진짜 개선인지** 세 방향으로 본다.

| 검증 | 묻는 것 | 왜 필요한가 |
| --- | --- | --- |
| 2-1 OOF 블록 부트스트랩 | 검증 구간 전체를 날짜 단위로 흔들어도 E13 > E12인가 | fold 5개만으로는 표본이 너무 적다 |
| 2-2 잡음 피처 기준선 | 같은 흐름의 **무의미한** 피처를 넣어도 이만큼 오르나 + 15개 중 최고를 고르면 얼마나 오르나 | 후보 ~15개 중 고른 **선택 편향** |
| 2-3 seed 반복 | RF 난수만 바꿔도 순위가 유지되나 | 모델 난수 잡음 |
| 2-4 시기별 OOF | 어느 시기에서 이기고 지는가 | 최근 구간 = Private에 가까움 |

**판정 기준 (미리 정함)** — 셋 다 만족하면 **E13**, 아니면 **E12**(같은 성능이면 단순한 쪽)
1. 2-1: OOF AUC 차이(E13−E12) > 0 일 확률 ≥ 90%
2. 2-2: '잡음 15개 중 최고' 기준선보다 관측 개선이 클 확률 p < 0.10
3. 2-3: seed 10개 중 8개 이상에서 E13 > E12 (CV 평균 기준)

### 2-1. OOF 예측 + 짝지은 블록 부트스트랩 (Private처럼 긴 구간을 한 번에 AUC로 계산)

In [ ]:
T1_NAMES = ['E8_t1_us_last', 'E11_t1_blend', 'E12_t1_blend', 'E13_t1_blend']
P = {nm: oof(REG[(nm, 'Target_1')][1], tr_(make_X(REG[(nm, 'Target_1')][0]))) for nm in T1_NAMES}
log_ = load_log().set_index(['name', 'target'])
display(pd.DataFrame({nm: {'CV 평균 (재계산)': fold_mean(P[nm]), 'CV 평균 (기록)': log_.loc[(nm, 'Target_1'), 'valid_mean'],
                           'OOF AUC (이어 붙임)': auc(P[nm][VA], yv[VA])} for nm in T1_NAMES}).T.round(4))

yva = yv[VA]
PAIRS = [('E13_t1_blend', 'E12_t1_blend'), ('E12_t1_blend', 'E11_t1_blend'), ('E13_t1_blend', 'E8_t1_us_last')]
B = 2000
boot_d = {pr: [] for pr in PAIRS}
for _ in range(B):
    i = block_idx(len(VA))
    for a_, b_ in PAIRS:
        boot_d[(a_, b_)].append(auc(P[a_][VA][i], yva[i]) - auc(P[b_][VA][i], yva[i]))
boot_tab = pd.DataFrame({f'{a_} - {b_}': {'관측 차이': auc(P[a_][VA], yva) - auc(P[b_][VA], yva),
                                          'CI 2.5%': np.percentile(d, 2.5), 'CI 97.5%': np.percentile(d, 97.5),
                                          'P(차이>0)': np.mean(np.array(d) > 0)} for (a_, b_), d in boot_d.items()}).T
display(boot_tab.round(4))

fig, ax = plt.subplots(1, len(PAIRS), figsize=(18, 3.8))
for a, ((a_, b_), d) in zip(ax, boot_d.items()):
    a.hist(d, bins=50, color='steelblue', alpha=.8); a.axvline(0, c='k', lw=1)
    a.axvline(np.mean(d), c='tomato', lw=2, label=f'평균 {np.mean(d):+.4f}')
    a.set_title(f'{a_[:3]} - {b_[:3]}: P(>0) = {np.mean(np.array(d) > 0):.0%}'); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

#### 👀 읽는 법
- 첫 표에서 **재계산 CV 평균 = 기록**이면 이 노트북의 모델이 `train.ipynb`와 같다는 확인이다.
- 히스토그램 대부분이 0(검은 선)의 오른쪽이면 기간을 어떻게 흔들어도 앞 모델이 낫다. 0을 넓게 걸치면 "차이를 확신할 수 없음".
- E12−E11, E13−E8은 비교용이다. 이미 채택한 개선들이 이 검정에서 어느 정도 확실했는지 보면 E13−E12의 위치를 가늠할 수 있다.

### 2-2. 잡음 피처 기준선 — 유가·키워드 심리 블록을 통째로 순환 이동시켜 E13을 다시 학습

In [ ]:
import time
X13 = tr_(make_X(REG[('E13_t1_blend', 'Target_1')][0]))
NEW = list(make_X(['oil_diff', 'kefin_swd']).columns)       # E13에서 새로 들어간 열
m13 = REG[('E13_t1_blend', 'Target_1')][1]
cv12, oof12 = fold_mean(P['E12_t1_blend']), auc(P['E12_t1_blend'][VA], yva)
obs_cv, obs_oof = fold_mean(P['E13_t1_blend']) - cv12, auc(P['E13_t1_blend'][VA], yva) - oof12

NREP = 60                                                    # ponytail: 60회 ≈ 1~2분. p를 더 촘촘히 보려면 200
t0 = time.time()
null = []
for k in rng.choice(np.arange(60, n - 60), NREP, replace=False):
    Xs = X13.copy(); Xs[NEW] = np.roll(X13[NEW].values, k, axis=0)   # 블록 내부 관계·자기상관 유지, Target과의 시점만 끊음
    p = oof(m13, Xs)
    null.append((fold_mean(p) - cv12, auc(p[VA], yva) - oof12))
null = np.array(null)
print(f'{NREP}회 ({time.time() - t0:.0f}초)')

N_CAND = 15                                                  # 실제로 LR 쪽에 시험해 본 후보 수(학습관리.md 기록)
max15 = np.array([[rng.choice(null[:, j], N_CAND).max() for _ in range(5000)] for j in range(2)]).T
res22 = pd.DataFrame({
    '관측 개선 (E13-E12)': [obs_cv, obs_oof],
    '잡음 1개 평균': null.mean(0), 'p (잡음 1개 대비)': [(1 + (null[:, j] >= o).sum()) / (1 + NREP) for j, o in enumerate([obs_cv, obs_oof])],
    '잡음 15개 중 최고 평균': max15.mean(0), 'p (15개 중 최고 대비)': [max((max15[:, j] >= o).mean(), 1 / (1 + NREP)) for j, o in enumerate([obs_cv, obs_oof])]   # 반복 수가 정하는 최소값 1/(NREP+1),
}, index=['CV 평균', 'OOF AUC'])
display(res22.round(4))

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
for j, (a, nm) in enumerate(zip(ax, ['CV 평균', 'OOF AUC'])):
    a.hist(null[:, j], bins=20, alpha=.6, density=True, label='잡음 블록 1개')
    a.hist(max15[:, j], bins=40, alpha=.5, density=True, color='orange', label=f'잡음 {N_CAND}개 중 최고')
    a.axvline(res22.iloc[j, 0], c='tomato', lw=2.5, label=f'관측 E13-E12 {res22.iloc[j, 0]:+.4f}')
    a.axvline(0, c='k', lw=.8); a.set_title(f'{nm} 개선: 잡음 기준선과 비교'); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

#### 👀 그림 읽는 법
- **파랑** = 의미 없는 피처(시점만 어긋난 같은 블록)를 넣었을 때의 개선. 0 근처가 정상이다. LR의 강한 정규화(C=0.01)가 잡음을 얼마나 눌러 주는지도 보인다.
- **주황** = 그런 잡음 후보 15개를 시험해 **가장 좋은 것**을 골랐을 때의 개선. 우리가 실제로 한 일(후보 탐색 후 최고 채택)의 '운만으로 얻는 몫'이다.
- **빨간 선이 주황 분포의 오른쪽 끝 밖**이면 선택 편향을 감안해도 진짜 개선이다. 주황 안쪽이면 E13의 개선은 "많이 찾아본 덕"일 수 있다.
- 한계: 실제 15개 후보는 서로 다른 변수였고, 여기서는 같은 블록을 시점만 바꿔 흉내 냈다. 근사적인 기준선으로 본다.

### 2-3. seed 반복 — RF 난수만 바꿔 E12·E13을 10번씩

In [ ]:
SEEDS = range(10)
X12 = tr_(make_X(REG[('E12_t1_blend', 'Target_1')][0]))
m12 = REG[('E12_t1_blend', 'Target_1')][1]
seed_rows = []
for s in SEEDS:
    p12 = oof(clone(m12).set_params(rf__randomforestclassifier__random_state=s), X12)
    p13 = oof(clone(m13).set_params(rf__randomforestclassifier__random_state=s), X13)
    seed_rows.append({'seed': s, 'E12 CV': fold_mean(p12), 'E13 CV': fold_mean(p13),
                      'E12 OOF': auc(p12[VA], yva), 'E13 OOF': auc(p13[VA], yva)})
seeds = pd.DataFrame(seed_rows).set_index('seed')
seeds['차이 CV'] = seeds['E13 CV'] - seeds['E12 CV']; seeds['차이 OOF'] = seeds['E13 OOF'] - seeds['E12 OOF']
display(seeds.round(4).T)

fig, ax = plt.subplots(figsize=(8, 4))
for k, c in enumerate(['차이 CV', '차이 OOF']):
    ax.scatter(np.full(len(seeds), k) + rng.uniform(-.08, .08, len(seeds)), seeds[c], s=40)
ax.axhline(0, c='k', lw=.8); ax.set_xticks([0, 1], ['CV 평균 차이', 'OOF 차이'])
ax.set_title(f'E13 - E12 (seed 10개): CV에서 E13 우위 {(seeds["차이 CV"] > 0).sum()}/10'); plt.show()

### 2-4. 시기별 OOF — 어느 시기에서 이기고 지는가 (150일 이동 창, 검증 구간만)

In [ ]:
W2 = 150
pos = np.arange(0, len(VA) - W2 + 1, 10)
rollP = {nm: np.array([auc(P[nm][VA][s:s + W2], yva[s:s + W2]) for s in pos]) for nm in ['E8_t1_us_last', 'E12_t1_blend', 'E13_t1_blend']}
x_ = VA[pos + W2 // 2]
fig, ax = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
for nm, c in zip(rollP, ['0.6', 'steelblue', 'tomato']):
    ax[0].plot(x_, rollP[nm], c=c, lw=2, label=nm)
ax[0].axhline(.5, c='k', ls=':', lw=.8); ax[0].axhspan(.5 - band, .5 + band, color='0.92')
ax[0].legend(); ax[0].set_title('150일 창 OOF AUC (회색 띠 = 잡음 95% 범위)')
d = rollP['E13_t1_blend'] - rollP['E12_t1_blend']
ax[1].bar(x_, d, width=8, color=np.where(d > 0, 'tomato', 'steelblue')); ax[1].axhline(0, c='k', lw=.8)
ax[1].set_title(f'E13 - E12 (빨강 = E13 우세, {np.mean(d > 0):.0%} 창)'); ax[1].set_xlabel('Time_step (창 중앙)')
plt.tight_layout(); plt.show()
print('E12·E13 OOF 예측 상관:', np.corrcoef(P['E12_t1_blend'][VA], P['E13_t1_blend'][VA])[0, 1].round(3),
      '| 최근 300일 OOF AUC  E12', round(auc(P['E12_t1_blend'][VA][-300:], yva[-300:]), 4), ' E13', round(auc(P['E13_t1_blend'][VA][-300:], yva[-300:]), 4))

## 3. 판정 — 미리 정한 기준을 그대로 적용

In [ ]:
def chk(ok): return '✅' if ok else '❌'
rows3 = []
for c in ['kefin_swd_med', 'KE_부동산_swd', 'KE_아파트_swd', 'SH1_sameWD_med', 'SH2_sameWD_med', 'KE_sameWD_med']:
    s = sig.loc[c]
    crit = [s['순환이동 p'] < .05, bool(s['CI가 0.5 제외']), bool(s['방향 일치']), ROLL_SHARE[c] >= .7]
    rows3.append({'신호': c, 'AUC': round(s.AUC, 3), '① p<0.05': chk(crit[0]), '② CI': chk(crit[1]), '③ 앞뒤 방향': chk(crit[2]),
                  '④ 창 70%': chk(crit[3]), '판정': '믿을 만함' if all(crit) else '우연 가능성 있음'})
kef_coef = coef.loc['kefin_swd_med'] if 'kefin_swd_med' in coef.index else coef.iloc[-1]
rows3.append({'신호': 'kefin_swd ⑤ LR 계수·미국장 고정', '판정':
              f"계수 부호 일치 {chk(np.sign(kef_coef).nunique() == 1)} (fold2~5 {chk(np.sign(kef_coef.iloc[1:]).nunique() == 1)}) · 미국장 3분위 모두 단조 증가 {chk(mono.all())}"})
print('③ 신호 신뢰도'); display(pd.DataFrame(rows3).set_index('신호').fillna(''))

c1 = boot_tab.loc['E13_t1_blend - E12_t1_blend', 'P(차이>0)'] >= .9
c2 = res22.loc['CV 평균', 'p (15개 중 최고 대비)'] < .10
c3 = (seeds['차이 CV'] > 0).sum() >= 8
print('④ E13 채택 기준')
display(pd.DataFrame({'기준': ['1. OOF 부트스트랩 P(E13>E12) ≥ 90%', '2. 잡음 15개 중 최고 대비 p < 0.10', '3. seed 10개 중 8개 이상 E13 우위'],
                      '값': [f"{boot_tab.loc['E13_t1_blend - E12_t1_blend', 'P(차이>0)']:.0%}", f"{res22.loc['CV 평균', 'p (15개 중 최고 대비)']:.3f}",
                            f"{(seeds['차이 CV'] > 0).sum()}/10"],
                      '통과': [chk(c1), chk(c2), chk(c3)]}).set_index('기준'))
print('→ 결론:', 'E13 유지' if c1 and c2 and c3 else 'E12로 되돌리는 것을 검토 (같은 성능이면 단순한 쪽)')

### 결과를 볼 때 주의
- 이 노트북의 기준은 **E13을 채택할 근거가 충분한가**를 묻는다. 기준을 못 넘었다고 E13이 더 나쁘다는 뜻은 아니다. "더 낫다고 확신할 수 없으니 단순한 E12"라는 판단이다.
- 반대로 기준을 넘었더라도 Public 100일 점수는 표준오차가 약 0.057이라, Public 결과로 이 판정을 뒤집지 않는다.
- ⑥(국면 변화: 최근 데이터 가중·학습 창)은 아래 §4에서 같은 OOF·블록 부트스트랩 도구로 검증한다.

## 4. ⑥ 국면 변화 — 최근 데이터만 쓰거나 최근에 가중을 주면 나아지나

**배경**: T1 30일 상승비율이 0.17~0.83으로 크게 오르내리고, §1-2에서 키워드 신호도 Time_step 170~450 구간에서 꺼져 있었다. 지금은 **과거 전체를 같은 무게로** 학습한다(expanding). 관계가 시기에 따라 바뀐다면 최근을 더 믿는 편이 Private(Train 바로 다음 200일)에 유리할 수 있다.

| 절 | 내용 |
| --- | --- |
| 4-1 진단 | 관계 자체가 시기에 따라 변하나? (200일 이동 창 LR 계수 + T1 상승비율 국면) |
| 4-2 실험 | E13 구조는 그대로, **학습 방식만** 바꿈: ① 최근 W일만 학습(sliding window) ② 지수 가중(반감기 h일 전 데이터 = 무게 절반) |
| 4-3 판정 | 미리 정한 기준 적용 + 통과 후보는 seed 반복 |

**판정 기준 (미리 정함)** — 후보가 6개라 여러 번 비교한 몫을 감안해 ④보다 엄격하게 둔다. 아래를 모두 만족하는 후보가 있으면 채택 검토, 없으면 **E13(expanding, 가중 없음) 유지**
1. OOF 블록 부트스트랩 P(후보 > E13) ≥ 95%
2. CV 평균−std ≥ E13
3. 최근 두 fold(4·5) 평균이 E13 이상 (Private에 가장 가까운 구간)
4. (1~3 통과 시) seed 10개 중 8개 이상 E13 우위

### 4-1. 진단 — 관계가 시기마다 변하는가 (200일 이동 창 LR 표준화 계수)

In [ ]:
CW, CSTEP = 200, 20
lr_cols_ = list(lr_X.columns)
rc_rows = []
for s in range(0, n - CW + 1, CSTEP):
    m = clone(lr_m).set_params(logisticregression__C=0.1).fit(lr_X.iloc[s:s + CW], y1.iloc[s:s + CW])   # 창이 짧아 C를 약간 풀어 계수 변화가 보이게
    rc_rows.append(pd.Series(m[-1].coef_[0], index=lr_cols_, name=s + CW // 2))
rc = pd.DataFrame(rc_rows)
rc_g = pd.DataFrame({'미국 지수 (3개 평균)': rc[[c for c in lr_cols_ if c.split('_')[1] in ('Sp500', 'Dow', 'Nasdaq')]].mean(axis=1),
                     '환율 (2개 평균)': rc[[c for c in lr_cols_ if 'FX' in c]].mean(axis=1),
                     '유가 (3개 평균)': rc[[c for c in lr_cols_ if c.split('_')[1] in ('Brent', 'WTI', 'Dubai')]].mean(axis=1),
                     '키워드 심리': rc['kefin_swd_med']})

fig, ax = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={'height_ratios': [1, 1.4]})
ax[0].plot(np.arange(n), y1.rolling(30).mean(), c='0.4', lw=1.2, label='T1 30일 상승비율 (과거 30일)')
ax[0].axhline(y1.mean(), c='k', ls=':', lw=.8); ax[0].set_ylim(0, 1); ax[0].legend(fontsize=9)
ax[0].set_title('T1 국면: 30일 상승비율')
for c in rc_g: ax[1].plot(rc_g.index, rc_g[c], lw=2, marker='o', ms=3, label=c)
ax[1].axhline(0, c='k', lw=.8); ax[1].legend(fontsize=9, ncol=4)
ax[1].set_title(f'{CW}일 이동 창 LR 표준화 계수 (C=0.1) — 선이 0을 넘나들면 관계 자체가 시기마다 바뀜')
for a in ax:
    for s in VA_START: a.axvline(s, c='tomato', ls='--', lw=.7)
ax[1].set_xlabel('Time_step (창 중앙)')
plt.tight_layout(); plt.show()
display(pd.DataFrame({'부호 유지 비율': (np.sign(rc_g) == np.sign(rc_g.mean())).mean(),
                      '계수 평균': rc_g.mean(), '계수 표준편차': rc_g.std()}).round(3))

#### 👀 읽는 법
- **미국 지수 계수가 내내 양수**면 핵심 관계는 안정적이다 → 최근 가중의 이득은 크지 않을 가능성이 높다(오래된 데이터도 같은 규칙을 가르쳐 줌).
- 계수가 0을 넘나드는 변수는 **국면 의존**이다. 이런 변수가 많을수록 최근 가중이 도움이 될 수 있지만, 창이 짧으면 표본이 줄어 추정이 흔들리는 비용도 커진다(4-2에서 이 둘의 균형을 잰다).
- 상승비율(위) 자체는 AUC와 직접 관련이 없다. AUC는 **같은 구간 안에서 순서**를 보므로, 국면이 바뀌어도 '미국장이 오르면 더 오른다'는 순서가 유지되면 괜찮다.

### 4-2. 학습 창 · 최근 가중 실험 (E13 구조 고정, 학습 방식만 변경)

In [ ]:
def fit_blend(model, X, y, w=None):
    # VotingClassifier(soft, 같은 비중)와 동일: 구성 모델을 각각 학습. 가중치는 각 Pipeline 마지막 단계에 전달
    out = []
    for _, est in model.estimators:
        e = clone(est)
        e.fit(X, y, **({f'{e.steps[-1][0]}__sample_weight': w} if w is not None else {}))
        out.append(e)
    return out

def oof_regime(model, Xd, window=None, half=None):
    p = np.full(n, np.nan)
    for tr, va in FOLDS:
        if window: tr = tr[-window:]                               # 최근 W일만
        w = None
        if half:
            w = 0.5 ** ((tr[-1] - tr) / half)                      # 학습 구간 마지막 날 = 1, h일 전 = 0.5
            w = w / w.mean()                                       # 평균 1로 맞춤 → LR 정규화 강도(C)가 바뀌지 않게
        ests = fit_blend(model, Xd.iloc[tr], y1.iloc[tr], w)
        p[va] = np.mean([e.predict_proba(Xd.iloc[va])[:, 1] for e in ests], axis=0)
    return p

VARIANTS = {'E13 (expanding, 가중 없음)': {}, '창 250일': dict(window=250), '창 400일': dict(window=400), '창 550일': dict(window=550),
            '반감기 120일': dict(half=120), '반감기 250일': dict(half=250), '반감기 500일': dict(half=500)}
t0 = time.time()
PR = {k: oof_regime(m13, X13, **v) for k, v in VARIANTS.items()}
print(f'{len(PR)}개 ({time.time() - t0:.0f}초)')
BASE = 'E13 (expanding, 가중 없음)'
assert abs(fold_mean(PR[BASE]) - fold_mean(P['E13_t1_blend'])) < 1e-9, '직접 학습한 E13이 VotingClassifier 결과와 다름'

fold_s = pd.DataFrame({k: [auc(p[va], yv[va]) for _, va in FOLDS] for k, p in PR.items()}, index=[f'fold{i + 1}' for i in range(5)]).T
boots = {k: [] for k in PR if k != BASE}
for _ in range(2000):
    i = block_idx(len(VA)); b = auc(PR[BASE][VA][i], yva[i])
    for k in boots: boots[k].append(auc(PR[k][VA][i], yva[i]) - b)
res4 = pd.DataFrame({'CV 평균': fold_s.mean(1), 'CV std': fold_s.std(1, ddof=0)})
res4['평균-std'] = res4['CV 평균'] - res4['CV std']
res4['OOF AUC'] = [auc(PR[k][VA], yva) for k in PR]
res4['최근 fold4·5'] = fold_s[['fold4', 'fold5']].mean(1)
res4['P(> E13)'] = [np.nan] + [np.mean(np.array(boots[k]) > 0) for k in boots]
display(pd.concat([res4, fold_s], axis=1).round(4))

fig, ax = plt.subplots(1, 2, figsize=(18, 4.5), gridspec_kw={'width_ratios': [1.3, 1]})
im = ax[0].imshow(fold_s.sub(fold_s.loc[BASE]), cmap='RdBu_r', vmin=-.04, vmax=.04, aspect='auto')
ax[0].set_xticks(range(5), fold_s.columns); ax[0].set_yticks(range(len(fold_s)), fold_s.index)
for i in range(len(fold_s)):
    for j in range(5): ax[0].text(j, i, f'{fold_s.iloc[i, j] - fold_s.loc[BASE].iloc[j]:+.3f}', ha='center', va='center', fontsize=8)
ax[0].set_title('fold별 AUC 차이 (후보 - E13, 빨강 = 후보 우세, 오른쪽 = 최근)'); plt.colorbar(im, ax=ax[0])
ax[1].boxplot([boots[k] for k in boots], vert=False, showfliers=False); ax[1].set_yticks(range(1, len(boots) + 1), list(boots))
ax[1].axvline(0, c='k', lw=.8); ax[1].set_title('OOF AUC 차이 (후보 - E13) 블록 부트스트랩')
plt.tight_layout(); plt.show()

#### 👀 읽는 법
- **fold1은 학습 기간이 151일뿐**이라 창 250·400·550은 E13과 같다(차이 0). 차이는 fold 2~5에서 생긴다.
- 창이 짧을수록 '최근 관계'를 더 따르지만 학습 표본이 줄어 흔들린다. 반감기 방식은 오래된 날을 버리지 않고 무게만 줄이므로 보통 더 완만하다.
- 상자 그림이 0의 오른쪽에 확실히 있어야 의미가 있다. 0을 걸치면 "최근 가중 효과를 확신할 수 없음".

### 4-3. 판정

In [ ]:
cand = res4.drop(index=BASE)
ok = pd.DataFrame({
    '1. P(>E13) ≥ 95%': cand['P(> E13)'] >= .95,
    '2. 평균-std ≥ E13': cand['평균-std'] >= res4.loc[BASE, '평균-std'],
    '3. fold4·5 ≥ E13': cand['최근 fold4·5'] >= res4.loc[BASE, '최근 fold4·5']})
ok['모두 통과'] = ok.all(axis=1)
display(ok.replace({True: '✅', False: '❌'}))

passed = list(ok.index[ok['모두 통과']])
if passed:
    best = cand.loc[passed, 'OOF AUC'].idxmax()
    wins = 0
    for s in range(10):
        ms = clone(m13).set_params(rf__randomforestclassifier__random_state=s)
        wins += fold_mean(oof_regime(ms, X13, **VARIANTS[best])) > fold_mean(oof_regime(ms, X13))
    print(f'4. seed 반복 — {best}: {wins}/10 에서 E13 우위', '✅' if wins >= 8 else '❌')
    print('→ 결론:', f'{best} 채택 검토 (train.ipynb 최종 학습에도 같은 창/가중 적용 필요)' if wins >= 8 else 'E13 유지')
else:
    print('→ 결론: 기준을 통과한 후보 없음 → E13(expanding, 가중 없음) 유지')

### ⑥ 결과를 볼 때 주의
- 기준을 통과하지 못한 경우 "최근 데이터가 쓸모없다"는 뜻이 아니다. **과거 전체로 배운 관계가 이미 최근에도 통하고 있어서, 최근을 더 믿어도 얻는 것이 잡음보다 작다**는 뜻이다(4-1에서 미국 지수 계수가 안정적이면 이 해석과 맞는다).
- 국면(상승비율 수준) 자체를 맞히려면 Target의 과거 값이 필요하다. 하지만 Test 구간에는 Target이 없어서 Private 200일 동안 쓸 수 없으므로 후보에서 제외했다.

## 5. Prophet 단서 — T1에 1년 주기가 있나 (좁힌 단일 가설)

**단서**: `02_Prophet_시계열분석.ipynb`에서 T1을 Prophet으로 직접 예측했을 때 1년 주기를 켜면 0.565, 끄면 0.492였다. 지금까지 T1 모델에는 계절 정보가 전혀 없다.
**의심할 이유**: 924일 = 1년 주기 약 2.5번뿐이고, 실제 날짜가 없어 해석도 불가능하다. 이 정도 횟수로는 "매년 반복"과 "그 시기에 우연히"를 가르기 어렵다.

**변수 `yearly`**: sin(2π·t/365.25), cos(2π·t/365.25), t = Time_step. 날짜만의 함수라 과거·미래 값을 쓰지 않는다(Leakage 없음). E13의 **LR 쪽에만** 추가한다(RF는 그대로).

**판정 기준 (미리 정함, 하나라도 못 넘으면 E13 유지)**
1. OOF 블록 부트스트랩 P(후보 > E13) ≥ 95%
2. **무작위 주기 기준선** p < 0.05 — 주기를 120~800일 사이에서 무작위로 바꾼(1년 근처 300~430일 제외) 느린 파동 40개로 같은 실험. 1년 주기가 '아무 느린 파동'보다 나아야 계절성이다. 그렇지 않으면 시기 흐름을 흉내 낸 것일 뿐이다.
   (위상을 무작위로 바꾸는 방식은 쓰지 않는다: sin·cos를 함께 넣으면 어떤 위상이든 같은 모델이 된다.)
3. 최근 fold 4·5 평균 ≥ E13
4. seed 10개 중 8개 이상 후보 우세

참고(기준 아님): 1년 차 · 2년 차 · 3년 차(일부)의 '같은 시기' 상승률이 비슷한 모양인지 눈으로 확인한다.

### 5-1. 진단 — 해마다 같은 시기에 같은 일이 반복되나

In [ ]:
t_full = full['Time_step'].values.astype(float)
BLOCKS['yearly'] = pd.DataFrame({'yr_sin': np.sin(2 * np.pi * t_full / 365.25), 'yr_cos': np.cos(2 * np.pi * t_full / 365.25)})
t_tr = t_full[is_tr]

doc = ((t_tr - 1) % 365.25 // (365.25 / 12)).astype(int)        # 1년을 12구간(약 30일)으로
cyc = ((t_tr - 1) // 365.25).astype(int)                         # 몇 번째 해 (0, 1, 2=일부)
rate = pd.DataFrame({'구간': doc, '해': cyc, 'y': yv}).pivot_table(index='구간', columns='해', values='y', aggfunc='mean')
cnt = pd.DataFrame({'구간': doc, '해': cyc}).value_counts().unstack()
r01 = rate[0].corr(rate[1]); r12 = rate[1].corr(rate[2])

fig, ax = plt.subplots(1, 2, figsize=(18, 4.5), gridspec_kw={'width_ratios': [1.4, 1]})
for c, col in zip(rate.columns, ['#9e9d98', '#2a78d6', '#eb6834']):
    ax[0].plot(rate.index, rate[c], marker='o', lw=2, c=col, label=f'{c + 1}년 차 (n={int(cnt[c].sum())})')
ax[0].axhline(yv.mean(), c='k', ls=':', lw=.8); ax[0].set_xticks(range(12), [f'{i + 1}' for i in range(12)])
ax[0].set_xlabel('1년 안의 위치 (약 30일 구간, 실제 월 아님)'); ax[0].set_ylabel('T1 상승률')
ax[0].set_title(f'해마다 같은 시기의 T1 상승률 — 1↔2년 차 상관 {r01:.2f}, 2↔3년 차 상관 {r12:.2f}'); ax[0].legend(fontsize=9)
yr_only = oof(make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=3000)), tr_(BLOCKS['yearly']))
ax[1].plot(np.arange(n), y1.rolling(60).mean(), c='0.6', lw=1.2, label='T1 60일 상승비율')
ax[1].plot(np.arange(n), yr_only, c='#2a78d6', lw=2, label='1년 주기만 쓴 LR의 검증 예측')
for s in VA_START: ax[1].axvline(s, c='tomato', ls='--', lw=.7)
ax[1].set_title(f'1년 주기만으로 예측: CV {fold_mean(yr_only):.3f}, OOF {auc(yr_only[VA], yva):.3f}'); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

#### 👀 읽는 법
- 왼쪽: 진짜 1년 주기라면 세 선이 **비슷한 모양**(같은 구간에서 같이 오르내림)이어야 한다. 상관이 0 근처나 음수면 해마다 다른 일이 일어난 것이다.
- 오른쪽: 1년 주기 **하나만으로** 미래 구간(검증)을 얼마나 맞히는지. Prophet의 0.565가 여기서도 재현되는지 본다.

### 5-2. E13 + yearly vs E13 — 부트스트랩 · 무작위 주기 기준선 · seed

In [ ]:
LR_YR = tuple(LR13) + ('yearly',)
blocks_yr = FEATS['Target_1'] + ['us_last'] + list(LR13) + ['yearly']
X_yr = tr_(make_X(blocks_yr))
m_yr = t1_blend2(LR_YR)
p_yr = oof(m_yr, X_yr)
p13 = P['E13_t1_blend']
fs = lambda p: [auc(p[va], yv[va]) for _, va in FOLDS]
cmp5 = pd.DataFrame({'E13': fs(p13), 'E13 + yearly': fs(p_yr)}, index=[f'fold{i + 1}' for i in range(5)]).T
cmp5['CV 평균'] = cmp5.iloc[:, :5].mean(1); cmp5['평균-std'] = cmp5['CV 평균'] - cmp5.iloc[:, :5].std(1, ddof=0)
cmp5['OOF'] = [auc(p13[VA], yva), auc(p_yr[VA], yva)]; cmp5['fold4·5'] = cmp5[['fold4', 'fold5']].mean(1)
display(cmp5.round(4))

d_boot = []
for _ in range(2000):
    i = block_idx(len(VA)); d_boot.append(auc(p_yr[VA][i], yva[i]) - auc(p13[VA][i], yva[i]))
P_win = np.mean(np.array(d_boot) > 0)

obs_gain = fold_mean(p_yr) - fold_mean(p13)
periods = rng.choice(np.r_[np.arange(120, 300), np.arange(431, 801)], 40, replace=False)   # ponytail: 40회 ≈ 40초, p 최소 1/41
t0 = time.time(); null5 = []
for T_ in periods:
    Xn = X_yr.copy()
    Xn['yr_sin'], Xn['yr_cos'] = np.sin(2 * np.pi * t_tr / T_), np.cos(2 * np.pi * t_tr / T_)
    null5.append(fold_mean(oof(m_yr, Xn)) - fold_mean(p13))
null5 = np.array(null5)
p_null = (1 + (null5 >= obs_gain).sum()) / (1 + len(null5))
print(f'무작위 주기 {len(periods)}개 ({time.time() - t0:.0f}초)')

seed_w = sum(fold_mean(oof(clone(m_yr).set_params(rf__randomforestclassifier__random_state=s), X_yr)) >
             fold_mean(oof(clone(m13).set_params(rf__randomforestclassifier__random_state=s), X13)) for s in range(10))

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].hist(d_boot, bins=50, color='#2a78d6', alpha=.8); ax[0].axvline(0, c='k', lw=1)
ax[0].set_title(f'OOF 차이 (E13+yearly - E13) 블록 부트스트랩: P(>0) = {P_win:.0%}')
ax[1].scatter(periods, null5, s=25, c='0.55', label='무작위 주기 파동')
ax[1].scatter([365.25], [obs_gain], s=90, c='#eb6834', zorder=3, label=f'1년 주기 {obs_gain:+.4f}')
ax[1].axhline(0, c='k', lw=.8); ax[1].axvspan(300, 430, color='0.93'); ax[1].set_xlabel('파동 주기 (일)'); ax[1].set_ylabel('CV 평균 개선')
ax[1].set_title(f'주기별 개선 — 1년 주기가 특별한가 (p = {p_null:.3f})'); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

chk5 = pd.DataFrame({'기준': ['1. P(후보 > E13) ≥ 95%', '2. 무작위 주기 기준선 p < 0.05', '3. fold4·5 ≥ E13', '4. seed 10개 중 8개 이상'],
                     '값': [f'{P_win:.0%}', f'{p_null:.3f}', f"{cmp5.loc['E13 + yearly', 'fold4·5']:.4f} vs {cmp5.loc['E13', 'fold4·5']:.4f}", f'{seed_w}/10'],
                     '통과': [P_win >= .95, p_null < .05, cmp5.loc['E13 + yearly', 'fold4·5'] >= cmp5.loc['E13', 'fold4·5'], seed_w >= 8]})
display(chk5.assign(통과=chk5.통과.map({True: '✅', False: '❌'})).set_index('기준'))
print('→ 결론:', 'E13 + yearly 채택 검토' if chk5.통과.all() else '기준 미달 → E13 유지 (1년 주기는 쓰지 않음)')

#### 👀 읽는 법
- 오른쪽 그림에서 주황 점(1년 주기)이 회색 점들(아무 느린 파동)보다 **확실히 위**에 있어야 '계절성'이다. 회색 점들도 비슷하게 오르면, 개선은 1년 주기 덕이 아니라 '느린 파동이 시기별 상승/하락 흐름을 우연히 따라간 것'이다. 그런 흐름은 Private 200일에서 반복된다는 보장이 없다.

## 6. Public에서 E8이 E13보다 높았던 이유 + 변동성에 따라 모델을 바꾸면?

Public 리더보드 T1: 1회차 E8 **0.564** > 3회차 E13 **0.556** (검증에서는 E13이 앞섬). Test 정답 없이, **검증 예측(OOF)과 Test 설명변수의 분포만** 써서 이유를 찾는다.

### 6-1. Public 구간은 어떤 시기였나 + 변동 큰 날에 두 모델은 어떻게 다른가

In [ ]:
P8 = oof(REG[('E8_t1_us_last', 'Target_1')][1], tr_(make_X(REG[('E8_t1_us_last', 'Target_1')][0])))
P13 = P['E13_t1_blend']
pv_ = pd.read_csv('test_private.csv', encoding='cp949')
US3 = ['FI_Sp500', 'FI_Dow', 'FI_Nasdaq']
lvl = pd.concat([full.loc[is_tr, ['Time_step'] + US3], pv_[['Time_step'] + US3]]).sort_values('Time_step').reset_index(drop=True)
dd = lvl[US3].diff(); us_abs_all = dd.mask(dd == 0).ffill(limit=3).abs().mean(axis=1).values       # 오늘(직전 거래일) 미국 지수 등락폭 — 예측 시점에 이미 아는 값
us_abs = us_abs_all[:n]; us_pv = us_abs_all[n:]
v = us_abs[VA]; thr_v = np.nanpercentile(v, 80); hi = v >= thr_v
seg = {'검증 770일': v >= thr_v, 'Public 100일': us_pv[:100] >= thr_v, 'Private 뒤 100일': us_pv[100:] >= thr_v, 'Private 200일': us_pv >= thr_v}
print('변동 큰 날(검증 상위 20% 경계) 비율:', {k: round(np.mean(x), 2) for k, x in seg.items()})
print(f'Train 최대 등락폭을 넘는 날: Public {np.sum(us_pv[:100] > np.nanmax(us_abs))}일, Private 뒤 100일 {np.sum(us_pv[100:] > np.nanmax(us_abs))}일')
display(pd.DataFrame({'E8 (RF 단독)': [auc(P8[VA][hi], yva[hi]), auc(P8[VA][~hi], yva[~hi])],
                      'E13 (RF + LR)': [auc(P13[VA][hi], yva[hi]), auc(P13[VA][~hi], yva[~hi])]},
                     index=[f'변동 큰 날 ({hi.sum()}일)', f'평범한 날 ({(~hi).sum()}일)']).round(3))

fig, ax = plt.subplots(1, 2, figsize=(18, 4.2))
ts = np.r_[np.arange(1, n + 1), pv_.Time_step.values]
ax[0].plot(ts, us_abs_all, c='0.5', lw=.8); ax[0].axhline(thr_v, c='#eb6834', ls='--', lw=1.2, label='변동 큰 날 경계 (검증 상위 20%)')
ax[0].axhline(np.nanmax(us_abs), c='k', ls=':', lw=1, label='Train 최대'); ax[0].axvspan(925, 1024, color='#fde3d6', label='Public'); ax[0].axvspan(1025, 1124, color='#e6eef9', label='Private 뒤 100일')
ax[0].set_title('오늘 미국 지수 등락폭 — Public 구간은 Train에 없던 크기의 변동이 몰림'); ax[0].legend(fontsize=8, loc='upper left'); ax[0].set_xlabel('Time_step')
x = np.arange(2); w = .38
ax[1].bar(x - w / 2, [auc(P8[VA][hi], yva[hi]), auc(P8[VA][~hi], yva[~hi])], w, color='#9e9d98', label='E8 (RF 단독)')
ax[1].bar(x + w / 2, [auc(P13[VA][hi], yva[hi]), auc(P13[VA][~hi], yva[~hi])], w, color='#2a78d6', label='E13 (RF + LR)')
ax[1].set_xticks(x, ['변동 큰 날', '평범한 날']); ax[1].axhline(.5, c='k', ls=':', lw=.8); ax[1].set_ylim(.4, .9); ax[1].legend(fontsize=9)
ax[1].set_title('검증 770일: 변동 큰 날엔 E8, 평범한 날엔 E13이 강함')
plt.tight_layout(); plt.show()

# E13 내부: Public에서 LR이 극단값을 직선으로 늘려 순위를 독점
m13f = clone(REG[('E13_t1_blend', 'Target_1')][1]); X13a = make_X(REG[('E13_t1_blend', 'Target_1')][0]); m13f.fit(X13a[is_tr], y1)
rf_p, lr_p = (m13f.named_estimators_[k].predict_proba(X13a[~is_tr])[:, 1] for k in ('rf', 'lr'))
rf_t, lr_t = (m13f.named_estimators_[k].predict_proba(X13a[is_tr])[:, 1] for k in ('rf', 'lr'))
print(f'E13 LR 확률 범위: Train {lr_t.min():.2f}~{lr_t.max():.2f} → Public {lr_p.min():.2f}~{lr_p.max():.2f} | '
      f'Public 최종 순위와의 상관: LR {pd.Series(rf_p + lr_p).rank().corr(pd.Series(lr_p).rank()):.2f}, RF {pd.Series(rf_p + lr_p).rank().corr(pd.Series(rf_p).rank()):.2f}')

### 6-2. 구간 구성에 맞춘 기대 AUC — Public 역전은 우연이었나, Private에서는?

In [ ]:
ih, il = np.where(hi)[0], np.where(~hi)[0]
def mix_auc(preds, share, B=2000, m=200):
    out = []
    for _ in range(B):
        nh = rng.binomial(m, share); idx = np.r_[rng.choice(ih, nh), rng.choice(il, m - nh)]
        out.append([auc(p[VA][idx], yva[idx]) for p in preds])
    return np.array(out)
rows6 = []
for tag in ['검증 770일', 'Public 100일', 'Private 200일']:
    r_ = mix_auc([P8, P13], np.mean(seg[tag]))
    rows6.append({'구성': tag, '변동 큰 날 비율': np.mean(seg[tag]), 'E8': r_[:, 0].mean(), 'E13': r_[:, 1].mean(), 'E13 - E8': (r_[:, 1] - r_[:, 0]).mean(), 'P(E13 > E8)': np.mean(r_[:, 1] > r_[:, 0])})
display(pd.DataFrame(rows6).set_index('구성').round(3))

#### 👀 읽는 법
- Public 구성(변동 큰 날 72%)에서 두 모델의 기대치가 같다면, 실제 Public에서 E8이 0.007 앞선 것은 **동전 던지기 수준**이다.
- Private은 앞 100일 = Public(극단), 뒤 100일은 비교적 잠잠하다. 그 구성으로 다시 잰 값이 Private 기대치다.
- 한계: Train 최대를 넘는 극단값 날은 검증 구간에 없어서, 그날 LR이 직선으로 늘려 잡는 위험은 이 추정에 들어 있지 않다.

### 6-3. 변동성에 따라 모델을 바꾸면? (전환 모델)
"변동이 큰 날"은 **미리 예측할 필요가 없다**. 기준이 *오늘 이미 마감된 미국장 등락폭*이라 예측 시점에 알고 있다.

**전환 기준 (미리 2개로 고정, 경계값은 매 fold 학습 구간의 상위 20%)**
| 후보 | 변동 큰 날 판정 | 그날 쓰는 모델 |
| --- | --- | --- |
| S1 | 오늘 미국 지수 등락폭 ≥ 학습 구간 80% 분위 | E8, 아니면 E13 |
| S2 | 최근 20일 등락폭 평균(위기 국면) ≥ 학습 구간 80% 분위 | E8, 아니면 E13 |

**판정 기준 (미리 정함)** — 둘 다 만족하면 채택 검토
1. Private 구성 기대 AUC에서 P(전환 > E13) ≥ 90%
2. 검증 구성 그대로에서 E13 대비 평균 손실 ≤ 0.005

In [ ]:
vol20_all = pd.Series(us_abs_all).rolling(20, min_periods=5).mean().values     # 과거 20일(오늘 포함) 평균 등락폭
def switch(sig):
    p = np.full(n, np.nan)
    for tr, va in FOLDS:
        th = np.nanpercentile(sig[tr], 80)                                        # 경계는 fold 학습 구간에서만
        p[va] = np.where(sig[va] >= th, P8[va], P13[va])
    return p
PS = {'E8': P8, 'E13': P13, 'S1 오늘 등락폭 전환': switch(us_abs), 'S2 20일 변동성 전환': switch(vol20_all[:n])}
rows7 = []
for k, p in PS.items():
    fa = [auc(p[va], yv[va]) for _, va in FOLDS]
    rows7.append({'모델': k, 'CV 평균': np.mean(fa), '평균-std': np.mean(fa) - np.std(fa), 'OOF': auc(p[VA], yva)})
res7 = pd.DataFrame(rows7).set_index('모델')
for tag in ['검증 770일', 'Private 200일']:
    r_ = mix_auc(list(PS.values()), np.mean(seg[tag]))
    res7[f'{tag} 기대 AUC'] = r_.mean(0)
    res7[f'{tag} P(> E13)'] = [np.mean(r_[:, j] > r_[:, 1]) for j in range(len(PS))]
display(res7.round(3))
c1_ = res7['Private 200일 P(> E13)'] >= .9
c2_ = (res7['검증 770일 기대 AUC'] - res7.loc['E13', '검증 770일 기대 AUC']) >= -.005
verdict = pd.DataFrame({'1. Private 구성 P(>E13) ≥ 90%': c1_, '2. 검증 구성 손실 ≤ 0.005': c2_}).drop(index=['E8', 'E13'])
display(verdict.replace({True: '✅', False: '❌'}))
print('→ 결론:', '전환 모델 채택 검토: ' + ', '.join(verdict.index[verdict.all(axis=1)]) if verdict.all(axis=1).any() else '기준 미달 → E13 유지')

#### 👀 읽는 법
- 전환 모델은 변동 큰 날엔 E8의 확률을, 평범한 날엔 E13의 확률을 쓴다. AUC는 모든 날을 한 줄로 세워 매기므로, 두 모델의 확률 수준이 다르면 섞는 과정에서 손해를 볼 수 있다(전환의 숨은 비용).
- 기준을 못 넘으면 "변동 큰 날에 E8이 강하다"는 관찰은 맞지만, **모델을 갈아 끼울 만큼의 이득은 없다**는 뜻이다.